In [1]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)
from jax_sn.operators.transport_operator import TransportOperatorBatched, TransportOperatorMultipleAnglesVmap, TransportOperatorVmap, TransportOperatorBatchedPallas, TransportOperatorMultipleAngles
from jax_sn.domain.geometric_domain import Domain, TopologicalSortedAngle
from jax_sn.domain.cross_sections import CrossSectionData
from jax_sn.solution_domain.basix_fem import BasixLagrangianSimplex
from jax_sn.operators.mass_operator import MassOperator
from jax_sn.solution_domain.solution_domain import SolutionDomain
from jax_sn.operators.moment_operator import MomentOperator, DiscreteOperator
from jax_sn.operators.scatter_operator import GatherScatterOperatorBroadcasted
from jax_sn.operators.multi_group_operator import MultiGroupOperatorScan
from jax_sn.operators.multi_group_operator import MultiGroupLowerTriangularBlockSolver
from jax_sn.tools.memory_analysis import MultiGroupOperatorSigmaAllParametrised, _filter_jit_f_to_memory, OperatorAnalysis, pytree_nbytes, MultiGroupOperatorEverythingParametrised

from jax_sn.domain.cross_sections import CrossSectionsIndexed
import equinox as eqx
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
import lineax as lx
import matplotlib.pyplot as plt
from dataclasses import dataclass
onp.set_printoptions(linewidth = 200)
from finite_elements.test_moment_scatter_operator import _setup_all_sb
from functions_for_testing import solve_and_plot

In [2]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain
from jax_sn.source import Source
from jax_sn.tools.memory_analysis import _filter_jit_f_to_memory
import lineax as lx

In [3]:
from finite_elements.test_multigroup_derivative   import modify_setup_mgop,get_rhs, _get_bb_verts_conn_xs_sources, base_region_sizes, base_resolutions, setup_from_verts, _setup_operators
def modify_setup_mgop(dim, degree, modify_verts_xs):
    verts, conn, cross_sections, sources = _get_bb_verts_conn_xs_sources(dim, base_region_sizes(), base_resolutions(), 5.0, 10)
    verts, cross_sections = modify_verts_xs(verts, cross_sections)

    solution_domain, sources, quadrature_set = setup_from_verts(verts, conn, cross_sections, sources, degree, 3)
    
    return _setup_operators(solution_domain, sources, quadrature_set)    

dim, degree = 2,2
bicgstab_solver = lx.BiCGStab(1e-12, 1e-12, max_steps = 160)
lower_triangular_solver = MultiGroupLowerTriangularBlockSolver(bicgstab_solver)
MGOP, source_eg0, _ = modify_setup_mgop(dim, degree, lambda v,c : (v,c))
rhs = get_rhs(MGOP, source_eg0)

W0115 13:15:38.692098   31352 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W0115 13:15:38.695090   31246 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.


In [4]:
def set_bs_to_1(MGOP):
    return eqx.tree_at(lambda op : op.transport_operator.transport_operator.batch_size, MGOP, replace = 1)


MGOP1 = set_bs_to_1(MGOP)


In [5]:
@eqx.filter_jit
def solve_and_sum(MGOP, rhs, solver):
    solution = lx.linear_solve(MGOP, rhs, solver)
    return jnp.sum(solution.value)



@eqx.filter_jit
def solve_result_everything(theta, MGOP, rhs_base, ltri_solver):    
    MGOP_new = MultiGroupOperatorEverythingParametrised.reconstruct_operator(MGOP, theta)
    solution_value =  lx.linear_solve(MGOP_new, rhs_base, solver=ltri_solver).value
    return jnp.sum(solution_value)

theta_everything_base = MultiGroupOperatorEverythingParametrised.theta_base(MGOP)
grad_theta_everything = eqx.filter_jit(jax.grad(solve_result_everything, argnums=0))
#df_dxi = grad_theta_everything(theta_everything_base, MGOP, rhs, lower_triangular_solver)

In [6]:
def print_memory_parametrisation(theta, MGOP, attribute_name, attribute_method):
    
    
    if attribute_name is None:
        op = MGOP
    else:
        op = getattr(MGOP, attribute_name)

    rhs = jnp.ones_like(op.in_structure())

    @eqx.filter_jit
    def sum_attr(mgop, theta, rhs, attribute_name):
        mgop1 = MultiGroupOperatorEverythingParametrised.reconstruct_operator(mgop, theta)
        if attribute_name is None:
            op = mgop1
        else:
            op = getattr(mgop1, attribute_name)

    
        solution = getattr(op, attribute_method)(rhs)
        
        return jnp.sum(solution)
    
    f_grad = eqx.filter_jit(jax.grad(sum_attr, argnums=1))
    
    print(f"{attribute_name}.{attribute_method} memory usage:\n\t Direct {_filter_jit_f_to_memory(sum_attr, MGOP, theta, rhs, attribute_name)} MB", f"\n\t Grad {_filter_jit_f_to_memory(f_grad, MGOP, theta, rhs, attribute_name)} MB")


print_memory_parametrisation(theta_everything_base, MGOP1, None, "mv")
print_memory_parametrisation(theta_everything_base, MGOP1, "moment_operator", "mv")
print_memory_parametrisation(theta_everything_base, MGOP1, "transport_operator", "inv")
print_memory_parametrisation(theta_everything_base, MGOP1, "transport_operator", "mv")
print_memory_parametrisation(theta_everything_base, MGOP1, "mass_operator", "mv")
#print_memory_parametrisation(theta_everything_base, MGOP1, "scatter_operator", "mv")
#print_memory_parametrisation(theta_everything_base, MGOP1, "moment_operator", "inv")



None.mv memory usage:
	 Direct {'peak': 12.2587890625, 'argument': 3.95343017578125, 'output': 3.0517578125e-05, 'temp': 8.309989929199219} MB 
	 Grad {'peak': 34.06144332885742, 'argument': 3.95343017578125, 'output': 0.01050567626953125, 'temp': 30.112342834472656} MB
moment_operator.mv memory usage:
	 Direct {'peak': 4.159461975097656, 'argument': 1.667724609375, 'output': 3.0517578125e-05, 'temp': 2.4919586181640625} MB 
	 Grad {'peak': 0.0105133056640625, 'argument': 0.0, 'output': 0.01050567626953125, 'temp': 3.0517578125e-05} MB
transport_operator.inv memory usage:
	 Direct {'peak': 4.064979553222656, 'argument': 2.265655517578125, 'output': 3.0517578125e-05, 'temp': 1.8028640747070312} MB 
	 Grad {'peak': 7.853672027587891, 'argument': 2.265655517578125, 'output': 0.01050567626953125, 'temp': 5.5890350341796875} MB
transport_operator.mv memory usage:
	 Direct {'peak': 3.4953765869140625, 'argument': 1.7104339599609375, 'output': 3.0517578125e-05, 'temp': 1.7879714965820312} MB 

In [73]:
from jax_sn.solution_domain.base_fem import FEMMatrices, PrecomputedGeometricData


def _mass_kernel_idx_in_precomp(
        elem_idx                    : int,
        fem_matrices                : FEMMatrices,    
        precomputed_geometric_data  : PrecomputedGeometricData,
        values                      : jnp.ndarray
    ):
    
    return  jnp.einsum("ij, ...j->...i", fem_matrices.mass_matrix , values[..., elem_idx, :] * precomputed_geometric_data.vol_jacobians_det[elem_idx])


@eqx.filter_jit
def _total_mass_kernel_idx_in_precomp(
        fem_matrices                : FEMMatrices,    
        precomputed_geometric_data  : PrecomputedGeometricData,
        values                      : jnp.ndarray
    ):
    '''
    Compute the mass matrix map
        v_i = ∑_{j} Jv ∫H_i H_j dV v_j 

    We index directly in the source instead of passing a subsource. This can save significant memory when vmapping over all elements and all directions with the same source independent of direction.

    Parameters
    ----------
    fem_matrices : FEMMatrices
        The finite element method matrices.
    precomputed_geometric_data : PrecomputedGeometricData
        The precomputed geometric data (Jacobian determinants and inverses).
    values : jnp.ndarray
        The source values array of shape [..., n_elements, n_basis]
    Returns
    -------
    values : jnp.ndarray [...., n_elements, n_basis]
        The computed value vector for all elements.
    '''            
    return jax.vmap(_mass_kernel_idx_in_precomp, in_axes=(0, None, None, None), out_axes=-2)(
                jnp.arange(precomputed_geometric_data.n_elements),
                fem_matrices,
                precomputed_geometric_data,
                values)

@eqx.filter_jit
def total_mass_kernel_2(fem_matrices : FEMMatrices,                         
                        precomputed_geometric_data : PrecomputedGeometricData,
                        values : jnp.ndarray):
    return jnp.einsum("ij,...ej,e->...e i", fem_matrices.mass_matrix , values, precomputed_geometric_data.vol_jacobians_det)



@eqx.filter_jit
def ree_sum(fem_matrices, precomputed_geometric_data, values):
    return _total_mass_kernel_idx_in_precomp(fem_matrices, precomputed_geometric_data, values).sum()

@eqx.filter_jit
@jax.checkpoint
def ree_sum2(fem_matrices, precomputed_geometric_data, values):
    return total_mass_kernel_2(fem_matrices, precomputed_geometric_data, values).sum()


print(MGOP1.mass_operator.in_structure())
rhs_m = jnp.ones_like(MGOP1.mass_operator.in_structure())

print(_filter_jit_f_to_memory(_total_mass_kernel_idx_in_precomp, MGOP1.transport_operator.solution_domain_data.fem_matrices, MGOP1.transport_operator.solution_domain_data.precomputed_geometric_data, rhs_m))


print(total_mass_kernel_2(
    MGOP1.transport_operator.solution_domain_data.fem_matrices,
    MGOP1.transport_operator.solution_domain_data.precomputed_geometric_data,
    rhs_m
).shape)
_filter_jit_f_to_memory(total_mass_kernel_2, MGOP1.transport_operator.solution_domain_data.fem_matrices, MGOP1.transport_operator.solution_domain_data.precomputed_geometric_data, rhs_m)
#_filter_jit_f_to_memory(total_mass_kernel_3, MGOP1.transport_operator.solution_domain_data.fem_matrices, MGOP1.transport_operator.solution_domain_data.precomputed_geometric_data, rhs_m)


ShapeDtypeStruct(shape=(72, 500, 6), dtype=float64)
{'peak': 6.59619140625, 'argument': 1.65203857421875, 'output': 1.6479721069335938, 'temp': 3.296875}
(72, 500, 6)


{'peak': 5.222625732421875,
 'argument': 1.65203857421875,
 'output': 1.6479721069335938,
 'temp': 1.9232177734375}

In [94]:
pytree_nbytes(rhs_m) / 1024**2

pytree_nbytes(rhs) / 1024**2

np.float64(3.2958984375)

In [ ]:
@eqx.filter_jit
def mass_ree(fem_matrices, precomputed_geometric_data, values):
    return jnp.einsum("...ej,e->...ej", values, precomputed_geometric_data.vol_jacobians_det)


@eqx.filter_jit
def mass_tot(fem_matrices, precomputed_geometric_data, values):
    return 
    values_scaled = mass_ree(fem_matrices, precomputed_geometric_data, values) #[...., e,j]
    return jnp.matvec(fem_matrices.mass_matrix[None, None, : , :] , values_scaled)
    return jnp.einsum("ij,...j->...i",fem_matrices.mass_matrix , values_scaled)  #[...., i]

_filter_jit_f_to_memory(mass_ree, MGOP1.transport_operator.solution_domain_data.fem_matrices, MGOP1.transport_operator.solution_domain_data.precomputed_geometric_data, rhs_m)
_filter_jit_f_to_memory(mass_tot, MGOP1.transport_operator.solution_domain_data.fem_matrices, MGOP1.transport_operator.solution_domain_data.precomputed_geometric_data, rhs_m)

{'peak': 6.59619140625,
 'argument': 1.65203857421875,
 'output': 1.6479721069335938,
 'temp': 3.296875}

In [74]:
%timeit _total_mass_kernel_idx_in_precomp(MGOP1.transport_operator.solution_domain_data.fem_matrices, MGOP1.transport_operator.solution_domain_data.precomputed_geometric_data,rhs_m).block_until_ready()

848 μs ± 7.03 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [75]:
%timeit total_mass_kernel_2(MGOP1.transport_operator.solution_domain_data.fem_matrices, MGOP1.transport_operator.solution_domain_data.precomputed_geometric_data,rhs_m).block_until_ready()

1.25 ms ± 21 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [76]:

print(_filter_jit_f_to_memory(ree_sum, MGOP1.transport_operator.solution_domain_data.fem_matrices, MGOP1.transport_operator.solution_domain_data.precomputed_geometric_data, rhs_m))

print(_filter_jit_f_to_memory(ree_sum2, MGOP1.transport_operator.solution_domain_data.fem_matrices, MGOP1.transport_operator.solution_domain_data.precomputed_geometric_data, rhs_m))

grad_1 = eqx.filter_jit(jax.grad(ree_sum, argnums=1))
grad_2 = eqx.filter_jit(jax.grad(ree_sum2, argnums=1))

print(_filter_jit_f_to_memory(grad_1, MGOP1.transport_operator.solution_domain_data.fem_matrices, MGOP1.transport_operator.solution_domain_data.precomputed_geometric_data, rhs_m))
print(_filter_jit_f_to_memory(grad_2, MGOP1.transport_operator.solution_domain_data.fem_matrices, MGOP1.transport_operator.solution_domain_data.precomputed_geometric_data, rhs_m))

{'peak': 6.596199035644531, 'argument': 1.65203857421875, 'output': 3.0517578125e-05, 'temp': 4.9445953369140625}
{'peak': 5.222633361816406, 'argument': 1.65203857421875, 'output': 3.0517578125e-05, 'temp': 3.5710601806640625}
{'peak': 6.638175964355469, 'argument': 1.648223876953125, 'output': 0.0458221435546875, 'temp': 4.944862365722656}
{'peak': 6.775238037109375, 'argument': 1.648223876953125, 'output': 0.0458221435546875, 'temp': 5.082069396972656}


In [27]:
@eqx.filter_jit
def total_mass_kernel_3(fem_matrices, precomputed_geometric_data, values):
    return jnp.einsum(
        "ij, ...ej -> ...ei",
        fem_matrices.mass_matrix,
        values * precomputed_geometric_data.vol_jacobians_det[None, :, None],
    )

_filter_jit_f_to_memory(total_mass_kernel_3, MGOP1.transport_operator.solution_domain_data.fem_matrices, MGOP1.transport_operator.solution_domain_data.precomputed_geometric_data, rhs_m)

{'peak': 6.59619140625,
 'argument': 1.65203857421875,
 'output': 1.6479721069335938,
 'temp': 3.296875}

In [ ]:
None.mv memory usage:
	 Direct {'peak': 1.2757568359375, 'argument': 0.4445343017578125, 'output': 3.0517578125e-05, 'temp': 0.8358688354492188} MB 
	 Grad {'peak': 3.4520912170410156, 'argument': 0.4445343017578125, 'output': 0.00693511962890625, 'temp': 3.0154190063476562} MB
moment_operator.mv memory usage:
	 Direct {'peak': 0.45157623291015625, 'argument': 0.1845703125, 'output': 3.0517578125e-05, 'temp': 0.2673492431640625} MB 
	 Grad {'peak': 0.0069427490234375, 'argument': 0.0, 'output': 0.00693511962890625, 'temp': 3.0517578125e-05} MB
transport_operator.inv memory usage:
	 Direct {'peak': 0.49428558349609375, 'argument': 0.2399139404296875, 'output': 3.0517578125e-05, 'temp': 0.25794219970703125} MB 
	 Grad {'peak': 0.8677101135253906, 'argument': 0.2399139404296875, 'output': 0.00693511962890625, 'temp': 0.6317596435546875} MB
transport_operator.mv memory usage:
	 Direct {'peak': 0.4156341552734375, 'argument': 0.1790771484375, 'output': 3.0517578125e-05, 'temp': 0.23963165283203125} MB 
	 Grad {'peak': 0.6078834533691406, 'argument': 0.1790771484375, 'output': 0.00693511962890625, 'temp': 0.4296112060546875} MB
mass_operator.mv memory usage:
	 Direct {'peak': 0.6617050170898438, 'argument': 0.1670074462890625, 'output': 3.0517578125e-05, 'temp': 0.4951324462890625} MB 
	 Grad {'peak': 0.6689910888671875, 'argument': 0.1670074462890625, 'output': 0.00693511962890625, 'temp': 0.495880126953125} MB

In [ ]:
TopologicalSortedAngle(levels=JitTracer<int64[50]>, angle=JitTracer<float64[3]>, levels_argsort=JitTracer<int64[50]>)
{'peak': 0.02783966064453125, 'argument': 0.0142822265625, 'output': 3.0517578125e-05, 'temp': 0.01697540283203125}
{'peak': 0.15105056762695312, 'argument': 0.0142822265625, 'output': 0.00693511962890625, 'temp': 0.13892364501953125}

In [15]:
@eqx.filter_jit
def solve_fwd(mgop, rhs, solver):
    return lx.linear_solve(mgop, rhs, solver=solver).value


@eqx.filter_jit
def solve_lower_tri(mgop, rhs, solver):
    ltri_solver=  MultiGroupLowerTriangularBlockSolver(solver)
    return lx.linear_solve(mgop, rhs, solver=ltri_solver).value

@eqx.filter_jit
def solve_bwd(mgop, rhs, solver):    
    return lx.linear_solve(mgop.transpose(), rhs, solver=solver).value

@eqx.filter_jit
def solve_bwd_lower_tri(mgop, rhs, solver):    
    ltri_solver=  MultiGroupLowerTriangularBlockSolver(solver)
    return lx.linear_solve(mgop.transpose(), rhs, solver=ltri_solver).value

def print_costs_solves(MGOP, rhs_base, bicgstab_solver):
    sgop = MGOP.to_single_group_operator(0)
    rhs_base = jnp.ones_like(MGOP.in_structure())
    bicgstab_solver = lx.BiCGStab(1e-12, 1e-12, max_steps = 160)
    print("Transport Operator")
    print(OperatorAnalysis.analyze_memory(sgop.transport_operator, "inv"))
    print("Single group operator")
    print(OperatorAnalysis.analyze_memory(sgop, "mv"))
    print("Multi group operator")
    print(OperatorAnalysis.analyze_memory(MGOP, "mv"))
    print("Forward solve")
    print(_filter_jit_f_to_memory(solve_fwd, MGOP, rhs_base, bicgstab_solver ))
    print("Forward tri solve")
    print(_filter_jit_f_to_memory(solve_lower_tri, MGOP, rhs_base, bicgstab_solver ))

    print("Backward solve")
    print(_filter_jit_f_to_memory(solve_bwd, MGOP, rhs_base, bicgstab_solver ))
    print("Backward tri solve")
    print(_filter_jit_f_to_memory(solve_bwd_lower_tri, MGOP, rhs_base, bicgstab_solver ))
print_costs_solves(MGOP1, rhs, bicgstab_solver)


Transport Operator
{'peak': 4.080230712890625, 'argument': 2.34576416015625, 'output': 1.6479721069335938, 'temp': 0.08850860595703125}
Single group operator
{'peak': 7.3433685302734375, 'argument': 1.575103759765625, 'output': 0.8239974975585938, 'temp': 4.946662902832031}
Multi group operator
{'peak': 12.287528991699219, 'argument': 4.047027587890625, 'output': 3.2959213256835938, 'temp': 4.947700500488281}
Forward solve
{'peak': 33.76802158355713, 'argument': 4.0999908447265625, 'output': 3.2959213256835938, 'temp': 28.027099609375}
Forward tri solve
{'peak': 18.936302185058594, 'argument': 4.0999908447265625, 'output': 3.2959213256835938, 'temp': 11.710212707519531}
Backward solve
{'peak': 39.007752418518066, 'argument': 4.0999908447265625, 'output': 3.2959213256835938, 'temp': 32.972900390625}
Backward tri solve
{'peak': 23.61260986328125, 'argument': 4.0999908447265625, 'output': 3.2959213256835938, 'temp': 17.09693145751953}


In [16]:
ltri_solver = MultiGroupLowerTriangularBlockSolver(bicgstab_solver)

In [17]:
from jax_sn.tools.memory_analysis import OperatorThetaParametrised
from jax_sn.solution_domain import PrecomputedGeometricData
from jax_sn.domain.cross_sections import CrossSectionData
from jax_sn.domain.geometric_domain import compute_face_normal_vectors_from_3d_simplices
from jax_sn.solution_domain.basix_fem import _precompute_jacobians_simplex

theta_everything_base = MultiGroupOperatorEverythingParametrised.theta_base(MGOP)

@eqx.filter_jit
def solve_result_everything(theta, MGOP, rhs_base, ltri_solver):    
    MGOP_new = MultiGroupOperatorEverythingParametrised.reconstruct_operator(MGOP, theta)
    solution_value =  lx.linear_solve(MGOP_new, rhs_base, solver=ltri_solver).value
    return jnp.sum(solution_value)

grad_theta_everything = eqx.filter_jit(jax.grad(solve_result_everything, argnums=0))

In [18]:
print("Everything parametrised")
theta_everything_base = MultiGroupOperatorEverythingParametrised.theta_base(MGOP)
print("Tri solver (batch 100)", _filter_jit_f_to_memory(solve_result_everything, theta_everything_base, MGOP, rhs, ltri_solver))
print("Tri solver (batch 1)", _filter_jit_f_to_memory(solve_result_everything, theta_everything_base, MGOP1, rhs, ltri_solver))
print("Total solver (batch 100 )", _filter_jit_f_to_memory(solve_result_everything, theta_everything_base, MGOP1, rhs, bicgstab_solver))
print("Total solver (batch 1)"  ,  _filter_jit_f_to_memory(solve_result_everything,theta_everything_base, MGOP1, rhs, bicgstab_solver))

print("\nGradients:")
print("Tri solver (batch 100)", _filter_jit_f_to_memory(grad_theta_everything, theta_everything_base, MGOP, rhs, ltri_solver))
print("Tri solver (batch 1)", _filter_jit_f_to_memory(grad_theta_everything, theta_everything_base, MGOP1, rhs, ltri_solver))
print("Total solver (batch 100 )", _filter_jit_f_to_memory(grad_theta_everything, theta_everything_base, MGOP, rhs, bicgstab_solver))
print("Total solver (batch 1)"  ,  _filter_jit_f_to_memory(grad_theta_everything, theta_everything_base, MGOP1, rhs, bicgstab_solver))

Everything parametrised
Tri solver (batch 100) {'peak': 20.616050720214844, 'argument': 3.9641876220703125, 'output': 3.0517578125e-05, 'temp': 17.87963104248047}
Tri solver (batch 1) {'peak': 18.86534881591797, 'argument': 3.9641876220703125, 'output': 3.0517578125e-05, 'temp': 15.072257995605469}
Total solver (batch 100 ) {'peak': 33.697068214416504, 'argument': 3.9641876220703125, 'output': 3.0517578125e-05, 'temp': 31.389389038085938}
Total solver (batch 1) {'peak': 33.697068214416504, 'argument': 3.9641876220703125, 'output': 3.0517578125e-05, 'temp': 31.389389038085938}

Gradients:
Tri solver (batch 100) {'peak': 39.479676246643066, 'argument': 3.9641876220703125, 'output': 0.01050567626953125, 'temp': 37.55375671386719}
Tri solver (batch 1) {'peak': 38.18897724151611, 'argument': 3.9641876220703125, 'output': 0.01050567626953125, 'temp': 37.537635803222656}
Total solver (batch 100 ) {'peak': 57.77612590789795, 'argument': 3.9641876220703125, 'output': 0.01050567626953125, 'temp'

In [ ]:
Everything parametrised
Tri solver (batch 100) {'peak': 6.010826110839844, 'argument': 0.455291748046875, 'output': 3.0517578125e-05, 'temp': 5.705070495605469}
Tri solver (batch 1) {'peak': 1.9462661743164062, 'argument': 0.455291748046875, 'output': 3.0517578125e-05, 'temp': 1.6901779174804688}
Total solver (batch 100 ) {'peak': 3.4295969009399414, 'argument': 0.455291748046875, 'output': 3.0517578125e-05, 'temp': 3.1511077880859375}
Total solver (batch 1) {'peak': 3.4295969009399414, 'argument': 0.455291748046875, 'output': 3.0517578125e-05, 'temp': 3.1511077880859375}

Gradients:
Tri solver (batch 100) {'peak': 12.493019104003906, 'argument': 0.455291748046875, 'output': 0.00693511962890625, 'temp': 12.087203979492188}
Tri solver (batch 1) {'peak': 3.8751726150512695, 'argument': 0.455291748046875, 'output': 0.00693511962890625, 'temp': 3.7803115844726562}
Total solver (batch 100 ) {'peak': 9.627723693847656, 'argument': 0.455291748046875, 'output': 0.00693511962890625, 'temp': 9.291305541992188}
Total solver (batch 1) {'peak': 5.754947662353516, 'argument': 0.455291748046875, 'output': 0.00693511962890625, 'temp': 5.325965881347656}

In [ ]:
Everything parametrised
Tri solver (batch 100) {'peak': 6.010826110839844, 'argument': 0.455291748046875, 'output': 3.0517578125e-05, 'temp': 5.705070495605469}
Tri solver (batch 1) {'peak': 1.9462661743164062, 'argument': 0.455291748046875, 'output': 3.0517578125e-05, 'temp': 1.6901779174804688}
Total solver (batch 100 ) {'peak': 3.4295969009399414, 'argument': 0.455291748046875, 'output': 3.0517578125e-05, 'temp': 3.1511077880859375}
Total solver (batch 1) {'peak': 3.4295969009399414, 'argument': 0.455291748046875, 'output': 3.0517578125e-05, 'temp': 3.1511077880859375}

Gradients:
Tri solver (batch 100) {'peak': 16.04212188720703, 'argument': 0.455291748046875, 'output': 0.00693511962890625, 'temp': 15.617095947265625}
Tri solver (batch 1) {'peak': 14.428302764892578, 'argument': 0.455291748046875, 'output': 0.00693511962890625, 'temp': 14.004547119140625}
Total solver (batch 100 ) {'peak': 16.042388916015625, 'argument': 0.455291748046875, 'output': 0.00693511962890625, 'temp': 15.618316650390625}
Total solver (batch 1) {'peak': 14.43014144897461, 'argument': 0.455291748046875, 'output': 0.00693511962890625, 'temp': 14.032379150390625}